# Add Cutouts to SGA data table

In [23]:
import os
import numpy as np
import fitsio
from astropy.io import fits
from astropy.table import Table
from tqdm import tqdm
import math

In [54]:
# Table Catalog
#sga2020 = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', hdu=1)
sgaBeta2025 = Table.read('/pscratch/sd/q/qshimp/SGA2025-data/sga2025_sizedTargets1000.csv')
#sgaAnchors = Table.read('/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152.fits')

# Directory Catalog
dirE = '/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Elliptical'
dirS = '/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Spiral'
dirL = '/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Lenticular'
dirI = '/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular'
dir = '/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/'

# Output file Catalog
#outAnch = "/pscratch/sd/q/qshimp/Sorter/152by152_sga2020_anchors.fits"
outTarg = "/pscratch/sd/q/qshimp/Sorter/152by152_sga2025_1000test.fits"
sgaBeta2025

col0,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam
int64,float64,float64,int64,float64,int64,float64,float64,float64
0,219.9258023922458,-32.53004546468686,4176048,17.78,0,70.35528,0.5832305,0.55550754
1,219.89199876009113,-32.34299358156585,4176016,17.71,0,150.42897,0.4607132,0.50789976
3,220.20921498175156,-32.0109310161516,243654,18.04,0,91.79849,0.7038806,0.507167
5,219.97219887120968,-31.718622738599922,4180680,17.14,0,92.536026,0.5446214,0.5610252
6,219.52960606217118,-31.677125640394838,245155,18.22,0,60.774433,0.20930941,0.5883865
10,219.94060790125502,-31.527629295847873,4168989,12.671,0,5.6187186,0.9850022,0.64752537
15,220.1967593743696,-31.466777588073057,4181392,13.37,0,160.23724,0.521736,0.6151214
16,220.2000823331581,-31.465108788558997,4172482,12.966,0,107.568214,0.6710952,0.48079902
18,220.4538075946316,-31.415308975937297,246182,17.05,0,45.81105,0.4843898,0.61828345


##### Path function

In [56]:
# Function to get RA slice directory name
def get_raslice(ra):
    return f'{int(ra):03d}'
def addPath(df, directory, output, ra="RA", dec="DEC", ID="SGA_ID", g=None, z=None, r=None, morphology=None, T_Type=None, M_Type=None, cutout_type="default"):    
    # Lists to store issues encountered
    processed_galaxies = []
    no_image_path = []
    resized = Table()
    ref_ids = resized["ref_id"] = df[ID]
    ras = resized["ra"] = df[ra]
    decs = resized["dec"] = df[dec]
    if g is not None:
        resized['g_mag'] = df[g]
    if z is not None:
        resized['z_mag'] = df[z]
    if r is not None:
        resized['r_mag'] = df[r]
    if morphology is not None:
        resized['Morphology'] = df[morphology]
    if T_Type is not None:
        resized["T_type"] = df[T_Type]
    if M_Type is not None:
        resized["Main_type"] = df[M_Type]
    paths = resized['Path'] = np.full(len(df), '', dtype=f'<U255')

    if cutout_type == "web":     
        for i in tqdm(range(len(df)), desc="Processing Galaxies", unit="galaxy", leave=True):
            ref_id = ref_ids[i]
            ra = ras[i]
            dec = decs[i]
    
            # Generate the path based on the provided directory structure
            path = os.path.join(directory, f'{ref_id}_grz_152_{ra:.4f}_{dec:.4f}.fits')
        
            if os.path.exists(path) and resized['Path'][i] == '':
                fits_file = fits.open(path, ignore_missing_simple=True, ignore_missing_end=True)
                image = fits_file[0].data
                fits_file.close()
                
                # Append filenames to new resized path column in resized table
                paths[i] = path
        
            else:
                no_image_path.append(path)
        resized['Path'] = paths
            
    else: # Default to cutouts generated in John's format
        for i in tqdm(range(len(df)), desc="Processing Galaxies", unit="galaxy", leave=True):
            ref_id = ref_ids[i]
            ra = ras[i]
            dec = decs[i]
    
            # Generate the path based on the provided directory structure
            path = os.path.join(datadir, get_raslice(ra), f'SGA2020-{sga_id:06d}.fits')
        
            if os.path.exists(path):
                fits_file = fits.open(path, ignore_missing_simple=True, ignore_missing_end=True)
                image = fits_file[0].data
                fits_file.close()
            
                # Append filenames to new resized path column in resized table
                paths[i] = path
        
            else:
                no_image_path.append(path)
        resized['Path'] = paths
            
    # Save the processed data to a new FITS file
    resized.write(output, overwrite=True)
    return no_image_path # Enables users to troubleshoot directly

##### Run History

In [57]:
# Run history    
no_image_path = addPath(sgaBeta2025, dir, outTarg, ra="Target_RA", dec="Target_DEC", ID="TargetID", r="mag", cutout_type="web")
#no_image_path = addPath(sgaAnchors, dirI, outAnch, g="G_MAG_SB26", z="Z_MAG_SB26", r="R_MAG_SB26", morphology="Morphology", T_Type='T_TYPE', M_Type='Main_Type', cutout_type="web")

Processing Galaxies: 100%|██████████| 1000/1000 [00:02<00:00, 438.23galaxy/s]


In [58]:
print(len(no_image_path))

0


In [62]:
# Completed tables
sga2025_152 = Table.read('/pscratch/sd/q/qshimp/Sorter/152by152_sga2025_1000test.fits')
sga2020_152 = Table.read("/pscratch/sd/q/qshimp/Sorter/152by152_sga2020_anchors.fits")
#sga2025_152
sga2020_152

ref_id,ra,dec,g_mag,z_mag,r_mag,Morphology,T_type,Main_type,Path
int64,float64,float64,float32,float32,float32,bytes4,float64,int64,bytes255
831151,213.50590133336513,31.14820277680165,17.112806,15.652968,16.261892,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/831151_grz_152_213.5059_31.1482.fits
841964,182.22624176312948,60.71909972151271,16.788475,15.38398,15.91633,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/841964_grz_152_182.2262_60.7191.fits
742784,144.67146870358894,51.82606783519872,15.109658,13.753175,14.197537,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/742784_grz_152_144.6715_51.8261.fits
1007845,183.18355501719242,31.45016405607238,16.120312,14.581675,15.226092,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/1007845_grz_152_183.1836_31.4502.fits
396832,200.3449984394637,4.316768068365768,17.439743,15.729994,16.446762,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/396832_grz_152_200.3450_4.3168.fits
390387,260.13342956053185,28.100807627116676,16.428516,14.9278345,15.554991,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/390387_grz_152_260.1334_28.1008.fits
1005162,172.20077047896189,39.58459111102822,17.856558,15.999874,16.569624,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/1005162_grz_152_172.2008_39.5846.fits
1114887,257.3376728569469,34.385891584617156,16.198465,14.789537,15.240975,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/1114887_grz_152_257.3377_34.3859.fits
218605,236.44655492342625,33.7305541769225,16.447487,14.988185,15.537429,E,-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular/218605_grz_152_236.4466_33.7306.fits
